# Phase 3 — Feature Engineering
**Goal:** Create 5 new smart features to improve model performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

df = pd.read_csv('../data/creditcard.csv')
print(f"Dataset loaded: {df.shape}")

In [ ]:
# Feature 1 — Hour of Day
# Fraud happens more at specific hours (late night)
df['hour_of_day'] = (df['Time'] / 3600 % 24).astype(int)
print("Feature 1 created: hour_of_day")
print(df['hour_of_day'].value_counts().sort_index().head())

In [ ]:
# Visualize fraud by hour
fig, ax = plt.subplots(figsize=(12,4))
fraud  = df[df['Class']==1]['hour_of_day'].value_counts().sort_index()
normal = df[df['Class']==0]['hour_of_day'].value_counts().sort_index()
ax.plot(normal.index, normal.values, label='Normal', color='steelblue', lw=2)
ax.plot(fraud.index,  fraud.values*100, label='Fraud x100', color='crimson', lw=2, linestyle='--')
ax.set_xlabel('Hour of Day'); ax.set_ylabel('Count')
ax.set_title('Transactions by Hour — Fraud vs Normal')
ax.legend(); ax.set_xticks(range(0,24))
plt.tight_layout()
plt.savefig('../reports/phase3_hour_analysis.png', dpi=150)
plt.show()

In [ ]:
# Feature 2 — Log Amount (reduces skewness)
df['log_amount'] = np.log1p(df['Amount'])
print("Feature 2 created: log_amount")

fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].hist(df['Amount'],     bins=100, color='steelblue', edgecolor='white')
axes[0].set_title('Amount — Original (Skewed)')
axes[1].hist(df['log_amount'], bins=100, color='crimson',   edgecolor='white')
axes[1].set_title('Log Amount — Normalized')
plt.tight_layout()
plt.savefig('../reports/phase3_log_amount.png', dpi=150)
plt.show()

In [ ]:
# Feature 3 — Amount Deviation (Z-score)
# How unusual is this amount compared to average?
df['amount_deviation'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
print("Feature 3 created: amount_deviation")
print("Normal avg deviation:", round(df[df['Class']==0]['amount_deviation'].mean(),4))
print("Fraud  avg deviation:", round(df[df['Class']==1]['amount_deviation'].mean(),4))

In [ ]:
# Feature 4 — V1 x V2 Interaction
df['v1_v2_interaction'] = df['V1'] * df['V2']
print("Feature 4 created: v1_v2_interaction")
print("Normal mean:", round(df[df['Class']==0]['v1_v2_interaction'].mean(),4))
print("Fraud  mean:", round(df[df['Class']==1]['v1_v2_interaction'].mean(),4))

In [ ]:
# Feature 5 — High Amount Flag (binary)
threshold = df['Amount'].quantile(0.90)
df['high_amount_flag'] = (df['Amount'] > threshold).astype(int)
print(f"Feature 5 created: high_amount_flag (threshold: {threshold:.2f})")
print("Fraud % in high amount:", round(df[df['high_amount_flag']==1]['Class'].mean()*100,3),"%")
print("Fraud % in normal amt :", round(df[df['high_amount_flag']==0]['Class'].mean()*100,3),"%")

In [ ]:
# Correlation heatmap
new_features = ['hour_of_day','log_amount','amount_deviation','v1_v2_interaction','high_amount_flag']
cols = new_features + ['V1','V2','V3','Amount','Class']
corr = df[cols].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../reports/phase3_correlation_heatmap.png', dpi=150)
plt.show()
print("Heatmap saved!")

In [ ]:
# Save engineered dataset and split
df_final = df.drop('Time', axis=1)
scaler2 = StandardScaler()
scale_cols = ['Amount','log_amount','amount_deviation','v1_v2_interaction']
df_final[scale_cols] = scaler2.fit_transform(df_final[scale_cols])

X = df_final.drop('Class', axis=1)
y = df_final['Class']
print(f"Total features after engineering: {X.shape[1]}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

joblib.dump(X_train_bal, '../data/X_train.pkl')
joblib.dump(X_test,      '../data/X_test.pkl')
joblib.dump(y_train_bal, '../data/y_train.pkl')
joblib.dump(y_test,      '../data/y_test.pkl')
joblib.dump(scaler2,     '../data/scaler.pkl')

print("All files saved! Phase 3 complete!")